System construction and test


In [ ]:
from datetime import date, datetime
import pandas as pd
import yfinance as yf
import time
#from Alert import Alert
pd.options.mode.chained_assignment = None  # default='warn'
#alert = Alert('1h','GGAL')

input:
    
     Titulo  : example: GGAL
     frequencia de tick : 1h (frequencia mais alta)
     dftitulo : vista do BD sqtitulosalpha.bd (testar ultimos periodas)
Parametros a variar para back testing ( definir rangos de variação)
     k, d, smooth (parametros do stch)
     dayM  (frequencia media) (multiplicador da frequencia mais alta) exemplo: 5
     semM  (frequencia baixa) (multiplicador da frequencia media) exemplo: 7

In [ ]:
dataini = '2013-04-03 19:30:00'
datafim = '2023-04-03 19:30:00'

In [ ]:
%%timeit
import sqlite3
import pandas as pd

# Caminho para o banco de dados
caminho_bd = r'C:\Users\scitr\anaconda_projects\Trading_System\Dados_Fontes\Alpha_Vantage\sqtitulosalpha.db'

# Conectando ao banco
conexao = sqlite3.connect(caminho_bd)

# Lendo a view
#consulta = 'SELECT * FROM vwtitulosdados ORDER BY datetime'
consulta = f"""
SELECT * FROM vwtitulosdados
WHERE datetime BETWEEN '{dataini}' AND '{datafim}'
ORDER BY datetime
"""

dftitulosdados = pd.read_sql_query(consulta, conexao)

# Fechando a conexão
conexao.close()

# Exibindo os primeiros registros para conferir
#display(dftitulosdados)
dftitulosdados = dftitulosdados.drop(columns=["symbol", "moeda", "intervalo","volume"])
#display(len(dftitulosdados))
#display(dftitulosdados.head(10))

In [ ]:

i = 'high'
K = 16  
D = 5    
smoth = 5  
dayM = 8  
semM = 4
stpl = 0.02


In [ ]:
#%%timeit
# Stochastic calculation
def stochastic(df, i, K, D, smoth):
        
    df["k"] = (100. * (df.close - df.low.rolling(K).min()) /
        (df.high.rolling(K).max() - df.low.rolling(K).min()))
    
    df["k" + i] = df.k.rolling(smoth).mean()
    df["d" + i] = df["k" + i].rolling(D).mean()
    
    df.drop(columns=["k"], inplace=True)  

    return df
dfstoch = stochastic(dftitulosdados, i, K, D, smoth)

#display (dfstoch.head(100))
#%time dfstoch

In [ ]:
#%%timeit
def set_test( df, intervalo, k, d, smth, dayM, semM):  
       

        df = stochastic(df, "high", k, d, smth)
        df = stochastic(df, "med", k*dayM, d*dayM, smth*dayM)
        df = stochastic(df, "low", k*dayM*semM, d*dayM*semM, smth*dayM*semM)
    
        return df

dfstoch_hml= set_test(dfstoch, i , K, D, smoth, dayM , semM )
#%time set_test(dfstoch, i , K, D, smoth, dayM , semM ) 
#display (dfstoch_hml.head(10))

In [ ]:
#%%timeit
#pd.set_option('display.max_rows', None)
df = dfstoch_hml
df["longbuylow"] = ((df["klow"] > 20) & (df["klow"] > df["dlow"])).astype(int)
df["longbuymed"] = ((df["kmed"] > 20) & (df["kmed"] > df["dmed"])).astype(int)
df["longbuyhigh"] = ((df["khigh"] > 20) & (df["khigh"] > df["dhigh"])).astype(int)



In [ ]:
#%%timeit
def long_buy_crits( df):   
   
    
    df["state"] = "standby"    
    
    
    for i in range(1, len(df)):        

        # long buy  states          

        if  df.loc[i, "longbuyhigh"] == 1 and df.loc[i, "longbuymed"] == 1 and df.loc[i, "longbuylow"] == 1 and df.loc[i-1, "state"] == "standby" :        
            df.loc[i, "state"] = "buylong"

            
            
        if  df.loc[i, "longbuymed"] == 1 and df.loc[i, "longbuylow"] == 1 and (df.loc[i-1, "state"] == "buylong" or df.loc[i-1, "state"] == "staylong") :
            df.loc[i, "state"] = "staylong" 

            
        if  df.loc[i, "longbuyhigh"] == 1 and df.loc[i, "longbuylow"] == 1 and df.loc[i, "longbuymed"] == 0 and (df.loc[i-1, "state"] == "buylong" or df.loc[i-1, "state"] == "staylong") :
            df.loc[i, "state"] = "staylong"

        # long sell state and price 
        
        if  (df.loc[i, "longbuylow"] == 0 ) and  (df.loc[i-1, "state"] == "buylong" or df.loc[i-1, "state"] == "staylong")  :
             df.loc[i, "state"] = "selllong"
            
            
        if  df.loc[i, "longbuyhigh"] == 0 and df.loc[i, "longbuymed"] == 0 and  (df.loc[i-1, "state"] == "buylong" or df.loc[i-1, "state"] == "staylong") :
             df.loc[i, "state"] = "selllong"
             
            
        if  (df.loc[i, "longbuylow"] == 0 or df.loc[i, "longbuymed"] == 0) and  df.loc[i-1, "state"] == "selllong"  :
             df.loc[i, "state"] = "standby" 
      
    return df
    
dfstate = long_buy_crits(dfstoch_hml) 

#%time long_buy_crits(dfstoch_hml)   
#display (df)

In [ ]:
dfstate = long_buy_crits(dfstoch_hml)
pd.set_option('display.max_rows', None)
dfbuysell = dfstate[(dfstate['state'] == 'buylong') | (dfstate['state'] == 'selllong')]
#df_intervalo = dfstate.iloc[800:1500] 
#display(df_intervalo)
display (len(dfbuysell))
display(dfbuysell)

In [ ]:
#%%timeit
import numpy as np

def long_buy_crits_numpy(df):
    n = len(df)
    state_array = np.full(n, "standby", dtype=object)  # inicializa com "standby"
    estado_anterior = "standby"

    high = df["longbuyhigh"].to_numpy()
    med = df["longbuymed"].to_numpy()
    low = df["longbuylow"].to_numpy()

    for i in range(1, n):
        if high[i] == 1 and med[i] == 1 and low[i] == 1 and estado_anterior == "standby":
            state_array[i] = "buylong"
            estado_anterior = "buylong"
        elif med[i] == 1 and low[i] == 1 and estado_anterior in ["buylong", "staylong"]:
            state_array[i] = "staylong"
            estado_anterior = "staylong"
        elif high[i] == 1 and low[i] == 1 and med[i] == 0 and estado_anterior in ["buylong", "staylong"]:
            state_array[i] = "staylong"
            estado_anterior = "staylong"
        elif low[i] == 0 and estado_anterior in ["buylong", "staylong"]:
            state_array[i] = "selllong"
            estado_anterior = "selllong"
        elif high[i] == 0 and med[i] == 0 and estado_anterior in ["buylong", "staylong"]:
            state_array[i] = "selllong"
            estado_anterior = "selllong"
        elif (low[i] == 0 or med[i] == 0) and estado_anterior == "selllong":
            state_array[i] = "standby"
            estado_anterior = "standby"
        else:
            state_array[i] = estado_anterior

    df["state"] = state_array
    return df

In [ ]:
dfstate = long_buy_crits_numpy(dfstoch_hml)
pd.set_option('display.max_rows', None)
dfbuysellop = dfstate[(dfstate['state'] == 'buylong') | (dfstate['state'] == 'selllong')]
#df_intervalo = dfstate.iloc[800:1500] 
#display(df_intervalo)
display (len(dfbuysell))
display(dfbuysell)

In [ ]:
#%%timeit
def Stop_Loss_Reentry (dfstate, stpl) :

    df = dfstate[(dfstate['state'] == 'buylong') | (dfstate['state'] == 'staylong')]
    df = df.reset_index(drop=True)
    df = df.drop(columns=["open" ,	"high" , "low" , "khigh","dhigh","kmed" ,"dmed","klow","dlow"])
    #drop(columns=["symbol", "moeda", "intervalo","volume"])
    df["stpl"] = 0.0
    stoplossprice = 0.0
    lastlongbuyprice = 0.0

    for i in range(0, len(df)):
        
        if df.loc[i, "state"] == "buylong" :
           stoplossprice = df.loc[i, "close"]
           df.loc[i,"stpl"] = df.loc[i, "close"] - stoplossprice * (1 - stpl)
           
        if df.loc[i, "state"]== "staylong" :
           df.loc[i, "stpl"] = df.loc[i, "close"] - stoplossprice * (1- stpl)
            
           if df.loc[i, "stpl"] < 0.0 :
                df.loc[i, "state"] = "selllong"
               
           if df.loc[i, "stpl"] > 0.0 and   (df.loc[i-1, "state"] == "selllong" or df.loc[i-1, "state"] == "sellstay") :
                df.loc[i, "state"] = "buylong"
               
           if df.loc[i, "stpl"] < 0.0 and   (df.loc[i-1, "state"] == "selllong" or df.loc[i-1, "state"] == "sellstay") :
                df.loc[i, "state"] = "sellstay"

    return df


In [ ]:
pd.set_option('display.max_rows', None)
dflongstpl = Stop_Loss_Reentry (dfstate, stpl)
display(len(dflongstpl))
df_intervalo = dflongstpl.iloc[0:300] 
display(df_intervalo)

In [ ]:
dflongsell = dfstate.drop(columns=["open" ,	"high" , "low" , "khigh","dhigh","kmed" ,"dmed","klow","dlow"])
dflongsell = dflongsell[(dflongsell['state'] == 'selllong')]
display (dflongsell)

dfbuysellstpl = dflongstpl[(dflongstpl['state'] == 'buylong') | (dflongstpl['state'] == 'selllong')]
df_intervalo = dfbuysellstpl.iloc[0:100]
#display(len(dfbuysell))
#display(df_intervalo)


In [ ]:
#%%timeit
# Suponha que df1 e df2 têm as mesmas colunas (incluindo 'datetime')
dflongbuysell = pd.concat([dflongsell, dfbuysellstpl], ignore_index=True)

# Ordenar pelo datetime (certifique-se de que é do tipo datetime)
dflongbuysell["datetime"] = pd.to_datetime(dflongbuysell["datetime"])
dflongbuysell = dflongbuysell.sort_values("datetime").reset_index(drop=True)
display (dflongbuysell)

In [271]:
#%%timeit
df = dflongbuysell

# Supondo que seu DataFrame se chame df
cond = (df["state"] == "selllong") & (df["stpl"].isna()) & (df["state"].shift(1) == "selllong")

dflongbuysell = df[~cond].reset_index(drop=True)
display (dflongbuysell.head(10))

,datetime,close,longbuylow,longbuymed,longbuyhigh,state,stpl
0,2013-09-06 12:00:00,5.6531,1,1,1,buylong,0.113062
1,2013-09-17 10:00:00,6.6434,1,0,0,selllong,NaN
2,2013-09-18 16:00:00,6.8149,1,1,1,buylong,0.136298
3,2013-09-30 10:00:00,7.2906,1,0,0,selllong,NaN
4,2013-10-10 11:00:00,7.9534,1,1,1,buylong,0.159068
5,2013-10-17 13:00:00,8.1561,1,0,0,selllong,NaN
6,2014-02-28 14:00:00,7.7272,1,1,1,buylong,0.154544
7,2014-03-10 16:00:00,7.9846,1,0,0,selllong,NaN
8,2014-03-20 11:00:00,8.4992,1,1,1,buylong,0.169984
9,2014-04-01 10:00:00,9.5596,1,0,0,selllong,NaN


Métricas

In [281]:
df = dflongbuysell
df ["index"] = 100.
for i in range(1, len(df)):       

                 

    if  df.loc[i, "state"] == "selllong" :
            df.loc[i, "index"] = ((df.loc[i,"close"]-df.loc[i-1,"close"])/df.loc[i-1,"close"]+1) * df.loc[i-1, "index"]
        
    if  df.loc[i, "state"] == "buylong" :
            df.loc[i, "index"] =  df.loc[i-1, "index"]


display (df)

,datetime,close,longbuylow,longbuymed,longbuyhigh,state,stpl,index
0,2013-09-06 12:00:00,5.6531,1,1,1,buylong,0.113062,100.000000
1,2013-09-17 10:00:00,6.6434,1,0,0,selllong,NaN,117.517822
2,2013-09-18 16:00:00,6.8149,1,1,1,buylong,0.136298,117.517822
3,2013-09-30 10:00:00,7.2906,1,0,0,selllong,NaN,125.720911
4,2013-10-10 11:00:00,7.9534,1,1,1,buylong,0.159068,125.720911
5,2013-10-17 13:00:00,8.1561,1,0,0,selllong,NaN,128.925028
6,2014-02-28 14:00:00,7.7272,1,1,1,buylong,0.154544,128.925028
7,2014-03-10 16:00:00,7.9846,1,0,0,selllong,NaN,133.219637
8,2014-03-20 11:00:00,8.4992,1,1,1,buylong,0.169984,133.219637
9,2014-04-01 10:00:00,9.5596,1,0,0,selllong,NaN,149.840743


In [317]:
def estatisticas_index(df):
    serie = df["index"].dropna()

    maximo = serie.max()
    minimo = serie.min()
    media = serie.mean()
    desvio = serie.std()
    taxagantot = (maximo - minimo)/minimo

    df["datetime"] = pd.to_datetime(df["datetime"])
    data_inicial = df["datetime"].min()
    data_final = df["datetime"].max()
    dias = (data_final - data_inicial).days
    

    taxagananualprom = (1 + taxagantot) ** (1 / dias * 365) - 1  
    taxalivrerisgoprom = 0.05
    sharpe = taxagananualprom/taxalivrerisgoprom
   
    return pd.Series({
        "Máximo": f"{maximo:.2f}",
        "Mínimo ": f"{minimo:.2f}",
        "Média": f"{media:.2f}",
        "Desvio padrão ": f"{desvio:.2f}",
        "Taxa de ganancia total ": f"{taxagantot * 100:.2f}%",
        "Taxa de ganancia anual prom ": f"{taxagananualprom * 100:.2f}%",
        "coef Sharpe": f"{sharpe:.2f}",
    })

In [319]:
resultado_index = estatisticas_index(df)
print(resultado_index.to_string())

Máximo                           341.51
Mínimo                           100.00
Média                            213.53
Desvio padrão                     53.53
Taxa de ganancia total          241.51%
Taxa de ganancia anual prom      13.97%
coef Sharpe                        2.79


In [257]:
def datas_drawdown_max(df):
    df = df.copy()
    df = df[df["index"].notna()]
    df["datetime"] = pd.to_datetime(df["datetime"])

    # Série com índice acumulado
    acumulado = df["index"]
    pico = acumulado.cummax()
    drawdown = acumulado - pico

    # Índice do drawdown máximo
    idx_vale = drawdown.idxmin()
    idx_pico = (acumulado[:idx_vale]).idxmax()

    # Datas correspondentes
    data_pico = df.loc[idx_pico, "datetime"]
    data_vale = df.loc[idx_vale, "datetime"]

    # Diferença percentual
    valor_pico = df.loc[idx_pico, "index"]
    valor_vale = df.loc[idx_vale, "index"]
    drawdown_pct = ((valor_vale - valor_pico) / valor_pico) * 100

    return pd.Series({
        "Data do Pico": data_pico.strftime("%Y-%m-%d %H:%M"),
        "Data do Vale": data_vale.strftime("%Y-%m-%d %H:%M"),
        "Valor do Pico": round(valor_pico, 2),
        "Valor do Vale": round(valor_vale, 2),
        "Drawdown Máximo (%)": f"{drawdown_pct:.2f}%"
    })

In [259]:
resultado_datas = datas_drawdown_max(df)
print(resultado_datas)

Data do Pico           2017-12-13 15:00
Data do Vale           2022-08-31 12:00
Valor do Pico                    264.73
Valor do Vale                    125.39
Drawdown Máximo (%)             -52.63%
dtype: object
